# Emotion data loading and preprocessing

Load the prepared three-class FER2013 dataset, check image/target batches, and show readable examples.
The targets are **Negative=0, Neutral=1, Positive=2**. Surprise is excluded.

Use the project's Python 3.13 environment with the CPU or CUDA dependency profile. This notebook
decodes and collates on CPU; it does not train a model or require GPU access.

Data and preprocessing settings are read from `configs/emotion.toml`. Run the notebook from the
repository or one of its subdirectories. Set `REPO_ROOT_OVERRIDE` below if it is opened elsewhere.
No source images, dataset splits or labels are changed.


In [1]:
from pathlib import Path
import sys, json
import matplotlib.pyplot as plt
import torch

REPO_ROOT_OVERRIDE = None
def find_repo(start):
    for candidate in (start, *start.parents):
        if (candidate / 'configs/emotion.toml').is_file() and (candidate / 'src/facial_attendance').is_dir():
            return candidate
    raise FileNotFoundError('Set REPO_ROOT_OVERRIDE to the repository checkout.')

if sys.version_info[:2] != (3, 13):
    raise RuntimeError('Select the project Python 3.13 environment.')
ROOT = find_repo(Path(REPO_ROOT_OVERRIDE or Path.cwd()).expanduser().resolve())
sys.path.insert(0, str(ROOT / 'src'))
from facial_attendance.emotion.data_loader import (
    CLASS_TO_ID, EmotionLayout, EmotionDataset, load_settings, make_loaders, tensor_transform)

settings = load_settings(ROOT / 'configs/emotion.toml', root=ROOT)
layout = EmotionLayout(settings.data_root)
loaders = make_loaders(layout, settings)
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('Model IDs:', CLASS_TO_ID)
print('Batch size:', settings.batch_size, '| seed:', settings.seed, '| augmentation:', settings.augmentation)


Python: 3.13.9 | PyTorch: 2.13.0+cpu
Model IDs: {'negative': 0, 'neutral': 1, 'positive': 2}
Batch size: 32 | seed: 42 | augmentation: none


## 1. Class counts and partitions

Counts are checked against the prepared manifest and summary. The loader also checks the actual file
inventory and requires unique pixel hashes across all partitions. Image hashes are checked again when
each image is loaded. These checks detect exact overlap; they do not establish label correctness.


In [2]:
print(f"{'Class':12s} {'Train':>8s} {'Validation':>11s} {'Test':>8s}")
for label in CLASS_TO_ID:
    print(f"{label:12s} {layout.counts['train'][label]:8,d} {layout.counts['val'][label]:11,d} {layout.counts['test'][label]:8,d}")
print('Totals:', {split: sum(counts.values()) for split, counts in layout.counts.items()})


Class           Train  Validation     Test
negative       10,089       2,521    3,293
neutral         3,858         964    1,224
positive        5,626       1,406    1,766
Totals: {'train': 19573, 'val': 4891, 'test': 6283}


## 2. Baseline preprocessing and real batches

Input is one 48×48 grayscale image. Convert pixels to float32 and divide by 255, giving values in `[0,1]`.
Batches contain images `[B,1,48,48]` and integer targets `[B]`. No mean/std normalization is applied.

Augmentation is disabled for this baseline on every partition. Training is shuffled with seed 42;
validation and test use fixed order. The last batch is kept even when it is smaller than the configured size.


In [3]:
for split, loader in loaders.items():
    images, targets = next(iter(loader))
    assert images.dtype == torch.float32 and tuple(images.shape[1:]) == (1, 48, 48)
    assert targets.dtype == torch.int64 and targets.shape == (images.shape[0],)
    assert torch.isfinite(images).all() and 0 <= float(images.min()) <= float(images.max()) <= 1
    assert ((targets >= 0) & (targets < 3)).all()
    print(split, '| images:', list(images.shape), str(images.dtype), '| targets:', list(targets.shape),
          str(targets.dtype), '| pixel range:', (float(images.min()), float(images.max())))
    print('First eight targets:', targets[:8].tolist())


train | images: [32, 1, 48, 48] torch.float32 | targets: [32] torch.int64 | pixel range: (0.0, 1.0)
First eight targets: [2, 2, 0, 1, 2, 2, 1, 0]
val | images: [32, 1, 48, 48] torch.float32 | targets: [32] torch.int64 | pixel range: (0.0, 1.0)
First eight targets: [1, 0, 0, 0, 2, 1, 1, 1]


test | images: [32, 1, 48, 48] torch.float32 | targets: [32] torch.int64 | pixel range: (0.0, 1.0)
First eight targets: [0, 2, 2, 0, 2, 0, 2, 2]


## 3. Image and target examples

Show one example of every target from each partition. The text uses the actual dataset target, not
the alphabetical position of a folder. Inspect whether the code's image/target alignment is correct;
the source annotation itself may still be imperfect.

These face previews are for local inspection. Clear this cell's image output before publishing the notebook.


In [4]:
fig, axes = plt.subplots(3, 3, figsize=(9, 9), layout='constrained')
for col, split in enumerate(('train', 'val', 'test')):
    dataset = EmotionDataset(layout, split, tensor_transform())
    for row, (label, expected_target) in enumerate(CLASS_TO_ID.items()):
        index = next(i for i, sample in enumerate(dataset.samples) if sample.label == label)
        image, target = dataset[index]
        assert target == expected_target
        axes[row, col].imshow(image[0].numpy(), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        axes[row, col].set_title(f'{split.title()} | {label.title()} | ID {target}', fontsize=10)
        axes[row, col].axis('off')
plt.show()


## 4. Repeatability

Validation/test preprocessing is deterministic. A fresh training loader with the same seed produces the
same first batch. Recreating a loader here tests repeatability; normal training should keep its loader
across epochs so the generator state advances and the epoch order can change.


In [5]:
first = make_loaders(layout, settings)
second = make_loaders(layout, settings)
for split in ('train', 'val', 'test'):
    a_images, a_targets = next(iter(first[split]))
    b_images, b_targets = next(iter(second[split]))
    assert torch.equal(a_images, b_images) and torch.equal(a_targets, b_targets)
    print(split, ': repeated batch matches')


train : repeated batch matches
val : repeated batch matches
test : repeated batch matches


## 5. Reproduce and hand over

Run `python scripts/check_emotion_data.py` with the project environment to save a small JSON report
under `reports/emotion/loader/`. It records real batches, examples from every class, versions, hashes,
class counts and verification results. See `docs/emotion-loader.md` for configuration and use in training.

This loader baseline is ready for model development. The GUI owner still needs to confirm face-crop
handling and use the same preprocessing when integrating the trained checkpoint. Independent review
of the loader is separate. No training accuracy, model performance or webcam integration is claimed here.
